# DiCo-NLI · AugB v5 · Commit 4（Kaggle T4 x2）

训练脚本 `train_augb_v5.py`（Claude：v4.1 加三臂，B0 的路径不变）。本次跑三臂，都只用原始英语数据、和 B0 配对：

- `OLat4`：O 改为 4 个分量的 logsumexp 头；
- `NLIinit`：编码器和池化层换成多语 NLI 检查点（`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`，固定 revision `b5113eb3…`）；
- `BiBCE`：CE 加两个方向的辅助 BCE（λ = 0.25）。

共 45 个 task；SMOKE 里 B0 复用检查不通过时自动加 B0 重跑 15 个。估计约 2.9 GPU 小时，T4 x2 墙钟约 1.5 小时；加 B0 重跑约 3.8 GPU 小时、墙钟约 2 小时。

**设置**：Accelerator 选 **GPU T4 x2**；右侧 Add Input 挂上本次的数据集（含 `train_augb_v5.py`）和官方仓库数据集；**打开 Internet**（要下载 NLI 检查点，约 560 MB）。

**第一次**：`RUN_FORMAL = False`，交互会话 Run All，只跑预检和 SMOKE，把打印出来的结果包发给 Claude。SMOKE 里有三项检查：三臂的数值检查和初始化检查、B0 复用检查、OLat1（K = 1）和 B0 逐值相同的检查。OLat1 那项不通过时，正式队列不会启动。

**正式**：`RUN_FORMAL = True`，Save Version → Save & Run All (Commit)。跑完在版本页 Output 下载 `augb_v5_c4_results.zip`。中断后把上一个版本的 Output 用 Add Input 挂上，再 Commit 一次，已完成的 task 自动跳过。


In [ ]:
RUN_FORMAL = False
COMMIT = "4"
import os, sys, glob, pathlib, subprocess, zipfile, shutil, json, time
os.environ["TOKENIZERS_PARALLELISM"] = "false"
ROOT = pathlib.Path('/kaggle/working/augb_v5')
OUT = pathlib.Path(f'/kaggle/working/augb_v5_c{COMMIT}_runs')
RESULT_ZIP = pathlib.Path(f'/kaggle/working/augb_v5_c{COMMIT}_results.zip')

def pack(note=''):
    if not OUT.exists():
        print('还没有结果目录', OUT); return
    tmp = RESULT_ZIP.with_suffix('.tmp'); n = 0
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.rglob('*')):
            if p.is_file() and p.suffix != '.pt':
                z.write(p, p.relative_to(OUT.parent)); n += 1
    tmp.replace(RESULT_ZIP)
    done = len(list(OUT.glob('tasks/**/done.json')))
    print(f'[结果包] {RESULT_ZIP}  {RESULT_ZIP.stat().st_size/1e6:.2f} MB，{n} 个文件；正式 task 完成 {done} {note}')
    try:
        from IPython.display import FileLink, display
        display(FileLink(str(RESULT_ZIP.relative_to('/kaggle/working'))))
    except Exception:
        pass
gpus = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip().splitlines()
print('RUN_FORMAL =', RUN_FORMAL, '| COMMIT =', COMMIT, '| GPU:', gpus)


In [ ]:
# transformers 钉在 Commit 1 用的 4.57.6，让 B0 复用检查尽量逐位复现；torch 用镜像自带的
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.57.6','sentencepiece>=0.2','protobuf>=4.25','scikit-learn>=1.3'], check=True)
import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
if torch.__version__ != '2.11.0+cu128':
    print('提醒：Commit 1 用的是 torch 2.11.0+cu128，版本不同时 B0 复用检查很可能不通过，B0 会自动重跑 15 项')


In [ ]:
# 训练包：Kaggle 会把上传的 zip 解压成文件夹；找同时有 train_augb_v5.py、data/gate.json、data_v4/c1_smoke_B0 的那个
hits = [p.parent for p in pathlib.Path('/kaggle/input').glob('**/train_augb_v5.py')
        if (p.parent/'data/gate.json').is_file() and (p.parent/'data_v3/mix_lists.json').is_file() and (p.parent/'data_v4/c1_smoke_B0/predictions.csv').is_file()
        and not (p.parent.parent/OUT.name).exists()]
if len(hits) != 1:
    raise RuntimeError('需要恰好一个训练包（含 train_augb_v5.py），找到：' + ', '.join(map(str, hits)))
if ROOT.exists(): shutil.rmtree(ROOT)
shutil.copytree(hits[0], ROOT)
print('Training package:', hits[0], '→', ROOT)


In [ ]:
def find_official():
    c = list(pathlib.Path('/kaggle/input').glob('**/final_data/train/dico_nli_train_track1_reference.csv'))
    if c: return c[0].parents[2]
    clone = pathlib.Path('/kaggle/working/SemEval-2027-Task-2-DiCo-NLI')
    if not clone.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/ilopezgazpio/SemEval-2027-Task-2-DiCo-NLI.git',str(clone)], check=True)
    return clone
OFFICIAL = find_official(); print('Official repo:', OFFICIAL)
BASE = [sys.executable, str(ROOT/'train_augb_v5.py'), '--bundle', str(ROOT), '--official', str(OFFICIAL), '--out', str(OUT), '--commit', COMMIT,
        '--model-revision', 'a0484667b22365f84929a935b5e50a51f71f159d']
subprocess.run(BASE + ['--stage', 'preflight'], check=True)


In [ ]:
# SMOKE（GPU 0）：三臂数值检查 → B0 / OLat1 / OLat4 / NLIinit / BiBCE 各两步 → 初始化检查 → B0 复用检查 → OLat1 = B0 检查；跑完马上打包
env = dict(os.environ, CUDA_VISIBLE_DEVICES='0')
try:
    subprocess.run(BASE + ['--stage', 'smoke'], check=True, env=env)
finally:
    for p in sorted((OUT/'smoke').glob('*/smoke.json')):
        s = json.loads(p.read_text())
        print(f"[SMOKE] {s['group']:8s} 更新 {s['updates']}  跳步 {s['skips']}  WF1 {s['validation_wf1']:.4f}  用时 {s['elapsed_sec']:.0f}s  {s['stable_environment']}")
    if (OUT/'arm_checks.json').exists():
        ac = json.loads((OUT/'arm_checks.json').read_text())['init']
        print('[初始化检查] OLat4 初始 p_O 和 B0 的最大差', f"{ac['K4_0.002']['max_abs_pO_diff']:.2e}", '| 零扰动', f"{ac['K4_zero']['max_abs_prob_diff']:.2e}",
              '| SentencePiece 相同', ac['spm_sha256']['b0'] == ac['spm_sha256']['nli'], '| B0 新初始化的键', ac['b0_newly_initialised_keys'])
    if (OUT/'b0_reuse.json').exists():
        br = json.loads((OUT/'b0_reuse.json').read_text())
        print('[B0 复用检查]', '通过：复用 Commit 1 的 B0，正式只跑 45 项' if br['status'] == 'pass' else '不通过：正式会多跑 B0 15 项', br['differences'])
    if (OUT/'olat1_check.json').exists():
        oc = json.loads((OUT/'olat1_check.json').read_text())
        print('[OLat1 = B0]', '通过' if oc['status'] == 'pass' else '不通过：正式队列不会启动', oc['differences'])
    pack('（SMOKE）')


In [ ]:
# 续跑：挂了上一次的 Output（文件夹或 zip）就恢复已完成的 task
dirs = sorted({p.parent.parent for p in pathlib.Path('/kaggle/input').glob(f'**/{OUT.name}/tasks/*')})
zips = [] if dirs else sorted(glob.glob(f'/kaggle/input/**/{RESULT_ZIP.name}', recursive=True))
if len(dirs) > 1 or len(zips) > 1: raise RuntimeError('找到多份旧结果，请只挂一份')
if dirs:
    shutil.copytree(dirs[0]/'tasks', OUT/'tasks', dirs_exist_ok=True, ignore=shutil.ignore_patterns('*.pt')); print('从文件夹恢复：', dirs[0])
elif zips:
    with zipfile.ZipFile(zips[0]) as z:
        names = [m for m in z.namelist() if pathlib.PurePosixPath(m).parts[:2] == (OUT.name, 'tasks')]
        for m in names:
            pp = pathlib.PurePosixPath(m)
            if pp.is_absolute() or '..' in pp.parts: raise ValueError(m)
        z.extractall('/kaggle/working', members=names)
    print('从 zip 恢复：', zips[0])
print('已有 done.json：', len(list(OUT.glob('tasks/**/done.json'))))


In [ ]:
# 正式：每张 GPU 一个 worker 进程，task 按估计用时分给两边；每 10 分钟打印一次进度；无论成败最后都打包
if RUN_FORMAL:
    n = max(1, len(gpus))
    (OUT/'logs').mkdir(parents=True, exist_ok=True)
    procs = []
    for i in range(n):
        log = open(OUT/'logs'/f'worker{i}.log', 'a')
        procs.append(subprocess.Popen(BASE + ['--stage', 'full', '--worker', str(i), '--workers', str(n)],
                                      stdout=log, stderr=subprocess.STDOUT, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))))
    t0 = time.time()
    try:
        while any(p.poll() is None for p in procs):
            time.sleep(600)
            print(f'[{(time.time()-t0)/3600:.1f} h] done.json：', len(list(OUT.glob('tasks/**/done.json'))), flush=True)
            for i in range(n):
                tail = (OUT/'logs'/f'worker{i}.log').read_text(errors='replace').strip().splitlines()[-2:]
                print(f'  worker{i}:', ' | '.join(tail)[-300:], flush=True)
    finally:
        print('worker 退出码：', [p.returncode for p in procs])
        pack('（正式）')
        shutil.rmtree(ROOT, ignore_errors=True)
    if any(p.returncode for p in procs):
        raise RuntimeError('有 worker 出错，见 logs/worker*.log（已打包）')
else:
    print('只跑了 SMOKE。把 RUN_FORMAL 改成 True，再 Save Version → Save & Run All (Commit)。')
